[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeffvan302/unsloth-embedding-colab/blob/main/embedding_finetune_poc.ipynb)

# Teaching an embedding model your vocabulary

This notebook takes a general-purpose embedding model, **Qwen3-Embedding**, and fine-tunes it on your own documents and word set, so that search and RAG find the right passage when people use *your* terms. It runs end to end on a **free Colab T4 GPU** in five steps, and finishes with a report showing what changed and by how much.

## Why fine-tune an embedding model?

An embedding model turns text into a list of numbers (a *vector*) so that texts with similar meaning get similar vectors. Search then means: embed the question, and return the passages whose vectors are closest.

Off-the-shelf models learned "similar meaning" from the public internet. They have never seen your product names, error codes, internal acronyms or procedures, so they fall back on surface word overlap:

- *"robot frozen, flashing amber"* may not find the passage about a **dwell lock**, because nothing tells the model those are the same thing;
- two internal terms that share a word (*brine cycle* and *cycle count*) may look related when they aren't.

Fine-tuning shows the model many examples of *"this question is answered by this passage"* from your own material, so its idea of similarity shifts towards what matters in your domain.

## How it works

| Step | What happens | Why |
|---|---|---|
| **▶ 1 · Set up** | Installs the tools, then loads your **data library**: documents (sample, upload or a Drive folder), an optional glossary, and topic groups. Documents are split into short passages. | Passages are the unit search returns, so they are the unit we train on. |
| **▶ 2 · Synthetic library** | A small local language model reads every passage and writes the questions a real user might ask about it. Your glossary becomes extra question → passage pairs. | Training needs thousands of question → answer examples, and almost nobody has them labelled. |
| **▶ 3 · Base model & baseline** | Holds out some passages for testing, loads the untrained model and measures how well it finds the right passage. | You can only claim an improvement against a measured starting point. |
| **▶ 4 · Train** | Snapshots how the model sees your vocabulary, picks tricky wrong answers, fine-tunes with LoRA, scores every epoch and keeps the best, then re-tests. | LoRA trains a small add-on (~1–2% of the weights), so it's fast and cheap and the base model stays intact. |
| **▶ 5 · Report** | Builds a single document: before vs after scores, the training curve, vocabulary maps and example searches. Downloads it with the trained model. | One place to see whether training helped, and to share it. |

## How to run it

1. **Runtime → Change runtime type → T4 GPU.**
2. Press ▶ on each step in order (or **Runtime → Run all**). Each step's settings are in the form next to it; double-click a step to see its code.
3. To use your own documents, change the settings in Step 1. Everything else adapts.

With the built-in sample, the whole run takes roughly 15–20 minutes, most of it the one-off install and question writing.

> **About the sample.** The built-in documents describe **Halyard**, a *fictional* warehouse-robotics product full of invented jargon (dwell locks, lantern codes, brine cycles…). Because the terms are made up, the base model can't already know them, which is exactly the situation with your own internal documents.

## ▶ Step 1 · Set up and load your data library

Installs Unsloth and its dependencies (skipped if already installed), checks the GPU, then loads your **data library**:

| Part | What it is | Where to set it |
|---|---|---|
| **Documents** | The text you want search to work on: `.md`, `.txt` or `.pdf` files. | `DOCS_SOURCE`: `sample`, `upload` (pick files when the step runs) or `folder` (e.g. a Google Drive folder, mounted automatically). |
| **Glossary / word set** | Your terms and what they mean. Each one becomes extra training examples and is tracked on the vocabulary map. | `GLOSSARY_CSV`: a CSV with columns `term,definition`. |
| **Topic groups** | Which documents belong together, used to colour the vocabulary map. | `DOC_GROUPS` in the step's code. Without it, each file is its own topic. |
| **Words to watch** | Extra terms to show on the vocabulary map. | `VOCAB_EXTRA` in the step's code. |

Each document is then split into **passages** of up to `CHUNK_MAX_WORDS` words, keeping paragraphs together. Passages are what search returns and what the model is trained on.

In [ ]:
# @title ▶ Step 1 · Set up and load your data library
# @markdown **Documents**: `sample` = built-in Halyard docs · `upload` = choose files · `folder` = read a folder
DOCS_SOURCE = "sample"  # @param ["sample", "upload", "folder"]
DOCS_FOLDER = "/content/drive/MyDrive/my_docs"  # @param {type:"string"}
# @markdown **Glossary** (optional): path to a CSV with columns `term,definition`
GLOSSARY_CSV = ""  # @param {type:"string"}
# @markdown **Embedding model** to fine-tune
BASE_MODEL = "unsloth/Qwen3-Embedding-0.6B"  # @param ["unsloth/Qwen3-Embedding-0.6B", "unsloth/Qwen3-Embedding-4B", "Qwen/Qwen3-Embedding-8B"] {allow-input: true}
# @markdown The instruction Qwen3-Embedding puts in front of every search query
TASK_INSTRUCTION = "Given a question about our internal documentation, retrieve the passage that answers it"  # @param {type:"string"}
# @markdown **Passage size** in words
CHUNK_MAX_WORDS = 80  # @param {type:"slider", min:40, max:300, step:10}

SEED = 42
QUERY_PROMPT = f"Instruct: {TASK_INSTRUCTION}\nQuery: "
OUT = "outputs"                      # everything the notebook produces ends up here
VIS_DIR = f"{OUT}/charts"

import os, re, gc, glob, json, math, random, csv, shutil, hashlib, importlib.util
from collections import Counter

# ---- 1. Install (only the first time in a session) ------------------------------------
if importlib.util.find_spec("unsloth") is None:
    print("Installing Unsloth and its dependencies - a few minutes the first time ...")
    from IPython.utils import io as _io
    with _io.capture_output() as _install_log:
        # Same install recipe as Unsloth's official Qwen3-Embedding notebook.
        if "COLAB_" not in "".join(os.environ.keys()):
            !pip install unsloth
        else:
            import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
            xformers = 'xformers==' + {'2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.35")
            if str(torch.version.cuda).startswith("13") and xformers.endswith("0.0.35"): xformers = "https://download.pytorch.org/whl/cu130/xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl"
            !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
            !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
            !pip install --no-deps --upgrade "torchao>=0.16.0"
        !pip install transformers==4.57.6
        !pip install --no-deps trl==0.22.2
        !pip install pypdf scikit-learn plotly
    importlib.invalidate_caches()
    if importlib.util.find_spec("unsloth") is None:
        print(_install_log.stdout[-5000:])
        raise RuntimeError("Unsloth did not install - see the log above.")

# ---- 2. Imports and GPU check ----------------------------------------------------------
import unsloth                                     # import first so its speed-ups are applied
from unsloth import FastSentenceTransformer, is_bf16_supported
import torch, transformers, sentence_transformers
import numpy as np, pandas as pd
from IPython.display import display, HTML
pd.set_option("display.max_colwidth", 100)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Use Runtime -> Change runtime type -> T4 GPU, then run this step again.")
GPU_NAME = torch.cuda.get_device_name(0)

# ---- 3. The data library ---------------------------------------------------------------
# Built-in sample: fictional Halyard documents and their glossary.

SAMPLE_DOCS = {
"fleet_overview.md": """Halyard runs two robot models on the warehouse floor. The Rook (model R-4) is a low, wheeled picking robot that carries totes between storage aisles and the packing stations. The Heron (model H-2) is a vertical lift robot that raises and lowers pallets in the high-bay racking. Both are coordinated by the Kestrel console, the browser-based control center used by shift supervisors.

A typical site runs 40 to 120 Rooks and 4 to 12 Herons. Rooks do the high-volume, small-item work; Herons only handle full pallets and never travel outside the high-bay zone. If a task needs both, Kestrel splits it into a Heron lift followed by a Rook tote handoff.""",

"dwell_lock.md": """A dwell lock is a safety state in which a robot stops and holds its exact position because two of its sensors disagree, for example the floor camera sees an obstacle that the lidar does not. While in dwell lock the robot keeps its brakes engaged and flashes amber. It will not move again on its own, even after the obstruction is gone.

To clear a dwell lock, a technician must physically walk to the robot, confirm the path is safe, and press the blue acknowledge button on its rear panel. Clearing it remotely from Kestrel is deliberately disabled. If the same robot enters dwell lock more than three times in one shift, Kestrel flags it for a sensor calibration before it can return to service.""",

"lantern_codes.md": """Lantern codes are the error codes reported by Heron lift robots. Every lantern code starts with L followed by three digits. Codes L-100 to L-199 are hydraulic faults, such as low fluid pressure or a slow mast extension. Codes L-200 to L-299 are load faults, meaning the pallet is overweight, off-center, or its weight changed during the lift.

Codes L-300 to L-399 cover positioning faults, where the Heron cannot confirm it is aligned with the rack beam. Codes L-400 and above are communication faults between the Heron and Kestrel. Any L-2xx load fault requires a human to inspect the pallet before the lift is retried; hydraulic and positioning faults can be retried once automatically.""",

"brine_cycle.md": """The brine cycle is the nightly battery recalibration that every Rook goes through. During the brine cycle the robot fully discharges to a safe floor level and then recharges slowly, which lets the battery management system re-learn the true capacity of the pack. It takes about 90 minutes and is scheduled automatically during the quietest hours of the night.

Skipping the brine cycle for more than five nights makes the reported battery percentage drift away from reality, and Rooks may then shut down mid-task with a charge reading of 20 percent or more. Supervisors can postpone a brine cycle from Kestrel, but never cancel it outright.""",

"cradle_swap.md": """A cradle swap is the procedure for replacing a Rook's battery cradle, the removable tray that holds its battery pack. Cradle swaps are needed when a battery is swollen, when the cradle latch is damaged, or when the pack has passed 1,500 charge cycles.

Before a cradle swap, put the Rook in burrow mode and wait for the status light to turn solid blue. Release the two side latches, slide the cradle out, and insert the replacement until both latches click. After the swap, the Rook must complete one full brine cycle before Kestrel will assign it work again.""",

"ghost_slots.md": """A ghost slot is a storage location that the inventory system believes is occupied but which is physically empty when a Rook arrives to pick from it. Ghost slots usually come from a missed scan during putaway or from an item that was moved by hand without being recorded.

When a Rook detects a ghost slot it reports the location to Kestrel, skips the pick, and the order is re-routed to another slot with the same item if one exists. Ghost slots are listed on the Kestrel exceptions page, and a stock controller should physically verify each one within 24 hours, because left alone they cause repeated failed picks.""",

"tally_drift.md": """Tally drift is the gradual difference between the stock count that Halyard records and the stock that is really on the shelves. Small amounts of tally drift are normal and come from damaged items, mis-scans, and ghost slots. Halyard reports drift per aisle as a percentage.

When tally drift in an aisle exceeds 2 percent, Kestrel schedules a cycle count for that aisle during the next quiet window. A cycle count sends a Rook with a camera mast down the aisle to re-scan every location. Drift above 5 percent is escalated to the site's inventory manager because it usually points to a process problem rather than random errors.""",

"pallet_echo.md": """A pallet echo happens when the same pallet ID is registered twice, usually because a pallet label was scanned at the dock and scanned again after being re-wrapped. Kestrel then believes there are two pallets in two different locations, and a Heron may be sent to lift a pallet that does not exist.

Kestrel automatically detects most pallet echoes by checking for identical IDs with arrival times less than one hour apart. Suspected echoes appear in amber on the high-bay map. To resolve one, a supervisor confirms which location really holds the pallet and retires the other record. Never re-label a pallet to fix an echo, as that hides the problem from the audit log.""",

"tote_handoff.md": """A tote handoff, written THO in logs, is the moment a Rook passes a tote onto a conveyor spur lane or to another robot. Spur lanes are the short conveyor branches that lead from the main conveyor loop to each packing station. A successful handoff is confirmed when the spur lane's photo-eye sees the tote and the Rook's load sensor reads empty.

Failed tote handoffs are almost always caused by a spur lane that is full or stopped. The Rook will wait at the spur for up to 60 seconds, then return the tote to a buffer location and report a THO timeout. Frequent THO timeouts at one station usually mean the packers there need more staff, not that the robots are faulty.""",

"quiet_window.md": """A quiet window is a scheduled period during which no software, firmware, or configuration changes may be applied to the Halyard fleet. Quiet windows are set around the busiest trading periods, such as the weeks before major holidays, and are configured by the site operations manager in Kestrel.

During a quiet window, firmware rollouts are paused automatically and configuration edits in Kestrel are read-only for everyone except users with the Site Owner role. Emergency fixes can still be applied, but they require a written approval from two Site Owners and are recorded in the change log with the reason.""",

"burrow_mode.md": """Burrow mode is a low-power parking state for Rooks. A Rook in burrow mode drives to the nearest parking bay, lowers its chassis onto the floor, switches off its drive motors and sensors except for a single heartbeat radio, and shows a solid blue light. Burrow mode is used overnight at sites with low volume, and before any maintenance work such as a cradle swap.

A Rook can be woken from burrow mode from Kestrel in about 30 seconds. Robots that stay in burrow mode for more than 14 days must be fully charged and run a brine cycle before they rejoin the fleet, because their battery calibration will be stale.""",

"wake_tokens.md": """A wake token is the one-time credential a new or repaired robot uses to join the fleet. Wake tokens are generated in Kestrel under Fleet, Add Robot, and are valid for 30 minutes. The technician enters the token on the robot's service screen, and the robot then downloads its site map, firmware, and safety zones.

Each wake token can be used only once and only for the robot model it was generated for, so a Heron token will be rejected by a Rook. If a token expires before the robot connects, simply generate a new one; expired tokens cannot be extended. Tokens are logged with the name of the user who created them.""",

"kestrel_roles.md": """Kestrel has four user roles. Viewers can see the live floor map and reports but cannot change anything. Supervisors can assign and pause work, clear exceptions such as ghost slots and pallet echoes, and postpone brine cycles. Technicians can put robots in burrow mode, generate wake tokens, and record maintenance such as cradle swaps.

Site Owners have every permission, including editing quiet windows, approving emergency changes, and managing other users. Roles are assigned per site, so a person can be a Site Owner at one warehouse and only a Viewer at another. Accounts that are not used for 90 days are automatically downgraded to Viewer.""",

"firmware_rings.md": """Halyard firmware is rolled out in rings so that a bad update never reaches the whole fleet at once. Ring 0 is a single designated test robot per site, ring 1 is ten percent of the fleet, and ring 2 is everyone else. Each ring must run for at least 48 hours without a regression before the next ring starts.

A regression is any rise in dwell locks, THO timeouts, or lantern codes compared with the previous week. If Kestrel detects a regression, the rollout halts and affected robots roll back to the last good firmware automatically. Rollouts never start or continue during a quiet window.""",

"escalation.md": """Halyard incidents are classified into three tiers. Tier 3 incidents affect a single robot and are handled by the on-site technician, for example a repeated dwell lock or a failed cradle swap. Tier 2 incidents affect a zone or a group of robots, such as all Herons in one high-bay aisle reporting the same lantern code, and must be reported to the regional support desk within one hour.

Tier 1 incidents stop or endanger the whole site, including any injury, a fire alarm, or the entire fleet losing contact with Kestrel. For tier 1, press the site emergency stop first, then call the 24-hour Halyard support line. Do not attempt to restart the fleet until support has confirmed it is safe.""",

"cold_storage.md": """Rooks can operate in chilled areas down to 0 degrees Celsius without changes. For freezer zones, down to minus 25 degrees Celsius, a site must use the cold-rated Rook variant, which has heated battery cradles and sealed wheel bearings. Standard Rooks that enter a freezer zone will be stopped by Kestrel at the zone boundary.

Cold-rated Rooks may spend at most 45 minutes inside a freezer zone before returning to a normal-temperature buffer area for 15 minutes to warm their batteries. Condensation can form on sensors when a robot leaves the freezer, so cold-rated Rooks automatically pause for two minutes at the exit to avoid false dwell locks.""",
}

# 🔌 HOOK — your word set. Term -> definition. Add, remove, or load from GLOSSARY_CSV.
GLOSSARY = {
    "dwell lock": "safety state where a robot freezes in place because two sensors disagree; cleared only by pressing the acknowledge button on the robot",
    "lantern code": "error code from a Heron lift robot, written L followed by three digits",
    "brine cycle": "nightly full discharge and slow recharge that recalibrates a Rook's battery",
    "cradle swap": "procedure for replacing the removable tray that holds a Rook's battery",
    "ghost slot": "storage location recorded as full that is physically empty",
    "tally drift": "difference between recorded stock and actual stock on the shelves",
    "pallet echo": "the same pallet ID registered twice, creating a phantom pallet",
    "tote handoff": "a Rook passing a tote onto a conveyor spur lane; abbreviated THO",
    "THO timeout": "a failed tote handoff because the spur lane was full or stopped",
    "spur lane": "short conveyor branch leading to a packing station",
    "quiet window": "change-freeze period when no firmware or configuration changes are allowed",
    "burrow mode": "low-power parking state for a Rook, shown by a solid blue light",
    "wake token": "one-time 30-minute credential a robot uses to join the fleet",
    "Kestrel": "the browser-based console used to control and monitor the Halyard fleet",
    "Rook": "Halyard's wheeled tote-picking robot, model R-4",
    "Heron": "Halyard's vertical pallet-lift robot, model H-2",
    "firmware ring": "stage of a firmware rollout: ring 0 test robot, ring 1 ten percent, ring 2 everyone",
}


# 🔌 Topic groups: document -> topic, used to colour the vocabulary map.
#    Documents not listed here use their own file name as the topic.
DOC_GROUPS = {
    "fleet_overview.md": "Fleet & console",       "kestrel_roles.md": "Fleet & console",
    "wake_tokens.md": "Fleet & console",
    "brine_cycle.md": "Battery & maintenance",    "cradle_swap.md": "Battery & maintenance",
    "burrow_mode.md": "Battery & maintenance",    "cold_storage.md": "Battery & maintenance",
    "dwell_lock.md": "Faults & incidents",        "lantern_codes.md": "Faults & incidents",
    "escalation.md": "Faults & incidents",
    "ghost_slots.md": "Inventory & flow",         "tally_drift.md": "Inventory & flow",
    "pallet_echo.md": "Inventory & flow",         "tote_handoff.md": "Inventory & flow",
    "quiet_window.md": "Change control",          "firmware_rings.md": "Change control",
} if DOCS_SOURCE == "sample" else {}

# 🔌 Extra words to watch on the vocabulary map (on top of the glossary and auto-detected phrases).
VOCAB_EXTRA = ["emergency stop", "support line", "change log", "emergency fixes", "firmware rollout",
               "photo-eye", "packing station", "battery cradle", "audit log", "exceptions page",
               "lidar"] if DOCS_SOURCE == "sample" else []

# 🔌 Searches shown before vs after in the report. Leave empty to use a few held-out questions.
EXAMPLE_QUERIES = ["robot stuck flashing amber and won't move", "what is an L-2xx error",
                   "how do I add a new robot to the fleet", "stock count doesn't match the shelf"
                   ] if DOCS_SOURCE == "sample" else []

# ---- 4. Load and chunk -----------------------------------------------------------------
def _mount_drive_if_needed(*paths):
    if any(str(p).startswith("/content/drive") for p in paths) and not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")

def read_file(path):
    if path.lower().endswith(".pdf"):
        from pypdf import PdfReader
        return "\n\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
    with open(path, encoding="utf-8", errors="ignore") as f:
        return f.read()

def load_documents():
    """🔌 Returns {name: text}. Replace with your own loader (Confluence, SharePoint, a database...)."""
    if DOCS_SOURCE == "sample":
        return dict(SAMPLE_DOCS)
    if DOCS_SOURCE == "upload":
        from google.colab import files
        print("Choose your .md, .txt or .pdf files:")
        return {name: read_file(name) for name in files.upload()}
    _mount_drive_if_needed(DOCS_FOLDER)
    paths = sorted(p for ext in ("md", "txt", "pdf")
                   for p in glob.glob(os.path.join(DOCS_FOLDER, "**", f"*.{ext}"), recursive=True))
    if not paths:
        raise FileNotFoundError(f"No .md, .txt or .pdf files found in {DOCS_FOLDER}")
    return {os.path.relpath(p, DOCS_FOLDER): read_file(p) for p in paths}

def load_glossary():
    glossary = dict(GLOSSARY) if DOCS_SOURCE == "sample" else {}
    if GLOSSARY_CSV.strip():
        _mount_drive_if_needed(GLOSSARY_CSV)
        with open(GLOSSARY_CSV.strip(), encoding="utf-8") as f:
            for row in csv.DictReader(f):
                glossary[row["term"].strip()] = row["definition"].strip()
    return glossary

def chunk_text(text, max_words=CHUNK_MAX_WORDS):
    """Pack paragraphs into passages of up to max_words; split overly long paragraphs on sentences."""
    paras = [re.sub(r"\s+", " ", p).strip() for p in re.split(r"\n\s*\n", text)]
    pieces = []
    for p in filter(None, paras):
        if len(p.split()) <= max_words:
            pieces.append(p)
            continue
        cur = []
        for s in re.split(r"(?<=[.!?])\s+", p):
            if cur and len(" ".join(cur + [s]).split()) > max_words:
                pieces.append(" ".join(cur)); cur = []
            cur.append(s)
        if cur: pieces.append(" ".join(cur))
    out, cur = [], ""
    for piece in pieces:
        if cur and len((cur + " " + piece).split()) > max_words:
            out.append(cur); cur = piece
        else:
            cur = (cur + "\n\n" + piece).strip()
    if cur: out.append(cur)
    return out

# Helpers for finding where a term lives (glossary pairs and the vocabulary map use them).
STOPWORDS = set("""a an the and or but if then than so of to in on at by for with from into onto over under
about after before during while within without between through per via as is are was were be been being it its
this that these those there their they them he she we you your our not no nor can cannot could should would will
may might must do does did done has have had having which who whom whose what when where why how all any each
every both either neither more most less least few many much some such only own same other others another also
just even very too again once up down out off one two three four five six seven eight nine ten first second new
use used uses using usually always never often automatically until etc
affect affects make makes need needs take takes keep keeps see sees set sets get gets mean means""".split())

def _words(text):
    return re.findall(r"[a-z][a-z0-9-]*", text.lower())

def _stem(w):
    return w[:-2] if w.endswith("es") and len(w) > 5 else (w[:-1] if w.endswith("s") and not w.endswith("ss") else w)

def _key(text):
    return " ".join(_stem(w) for w in _words(text))

def home_chunk(term):
    """The passage that uses the term most: the whole phrase counts most, then its words.
    A document whose file name contains the term (e.g. kestrel_roles.md for "Kestrel") gets a bonus."""
    t = term.lower()
    words = [w for w in _words(t) if w not in STOPWORDS]
    def score(c):
        text, name = c["text"].lower(), c["doc"].lower()
        return (10 * text.count(t) + sum(text.count(w) for w in words)
                + 15 * sum(_stem(w) in name for w in words if len(w) > 3))
    return max(chunks, key=score)

docs = load_documents()
chunks = [{"id": f"{name}#{i}", "doc": name, "text": c}
          for name, text in docs.items() for i, c in enumerate(chunk_text(text))]
glossary = load_glossary()
os.makedirs(VIS_DIR, exist_ok=True)

print(f"GPU: {GPU_NAME} | torch {torch.__version__} | transformers {transformers.__version__} "
      f"| sentence-transformers {sentence_transformers.__version__}\n")
display(pd.DataFrame({
    "documents": [len(docs)], "passages": [len(chunks)],
    "avg words per passage": [round(np.mean([len(c["text"].split()) for c in chunks]))],
    "glossary terms": [len(glossary)], "topic groups": [len(set(DOC_GROUPS.values())) or "one per file"],
}).set_axis(["data library"]))
print("Example passage:\n  " + chunks[min(1, len(chunks) - 1)]["text"][:300])

## ▶ Step 2 · Build the synthetic training library

**Why it's needed.** An embedding model learns from pairs: *this question → this passage is the answer*. Fine-tuning needs hundreds to thousands of them, written the way real people search. Hardly anyone has that labelled, and writing it by hand is slow. So we have a language model play the user.

**How it's built**

1. **A question writer reads each passage.** A small instruction-following model (`Qwen2.5-3B-Instruct`, which fits next to everything else on a T4) is shown one passage at a time and asked for `QUERIES_PER_CHUNK` different questions that the passage answers. The prompt asks for a mix of styles, because real searches come in all of them:
   - short keyword searches: *"L-2xx fault"*
   - full questions: *"What does a lantern code in the 200s mean?"*
   - **symptom descriptions that never name the term**: *"pallet lift stopped, says weight changed"*. These teach the model to connect everyday wording to your jargon, which is where off-the-shelf models fail most.

   It is also told not to copy phrases from the passage, so the model can't succeed by matching words alone.
2. **Cleaning.** Numbering and quotes are stripped, very short or long lines dropped, and duplicates removed. `QUESTION_ROUNDS` > 1 runs the writer again with different random choices for extra variety.
3. **Your glossary joins in.** Each term becomes several pairs with simple templates (*"define dwell lock"*, *"What does dwell lock mean?"*), pointing at its definition. In Step 3, each term is also paired with the passage that uses it most, so it's learned *in context*.
4. **Saved as a library.** Everything is written to `synthetic_library_*.jsonl` (one question per line, with the passage it belongs to). Open it from the Files panel to review or hand-edit. The file is reused until you tick `REGENERATE`, or change the documents or settings.

**How it's used later.** The library is split by passage: most of it trains the model, while the questions for held-out passages become the validation and test sets the model is measured on. It also drives *hard negatives* in Step 4: for each question, the wrong passage the base model likes most becomes a deliberate trap to train against.

**Quality matters more than quantity.** A stronger question writer, a few real user questions, or a reviewed library will do more than doubling the count. To use a hosted model instead, replace `generate_queries()`: it takes a list of passages and returns a list of question lists.

In [ ]:
# @title ▶ Step 2 · Build the synthetic training library
# @markdown **Question writer** (a local instruct model; larger ones need more GPU memory)
GENERATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"  # @param ["Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-7B-Instruct"] {allow-input: true}
# @markdown **Questions per passage**, and how many **rounds** to run the writer for more variety
QUERIES_PER_CHUNK = 8  # @param {type:"slider", min:2, max:20, step:1}
QUESTION_ROUNDS = 1  # @param {type:"slider", min:1, max:5, step:1}
# @markdown Tick to discard the saved library and write new questions
REGENERATE = False  # @param {type:"boolean"}

GEN_SYSTEM = ("You write realistic search queries that operators, technicians and support staff "
              "type when looking for information in internal documentation.")
GEN_USER = """Here is a passage from our documentation:

\"\"\"{chunk}\"\"\"

Write {n} different questions or search queries that this passage answers.
Vary them: some short keyword-style queries, some full questions, and some that describe a symptom or situation instead of naming the term.
Do not copy long phrases from the passage. Output one per line with no numbering and nothing else."""

def clean_lines(raw, n):
    out = []
    for line in raw.splitlines():
        line = re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", line).strip().strip('"').strip()
        if 8 <= len(line) <= 200 and line.lower() not in {o.lower() for o in out}:
            out.append(line)
    return out[:n]

def generate_queries(chunk_texts, n=QUERIES_PER_CHUNK, rounds=QUESTION_ROUNDS, batch_size=8):
    """🔌 Passages in, a list of questions per passage out. Swap in a hosted LLM here if you like."""
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(GENERATOR_MODEL, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL, torch_dtype=torch.float16, device_map="cuda")
    prompts = [tok.apply_chat_template(
                   [{"role": "system", "content": GEN_SYSTEM},
                    {"role": "user", "content": GEN_USER.format(chunk=c, n=n)}],
                   tokenize=False, add_generation_prompt=True) for c in chunk_texts]
    results = [[] for _ in chunk_texts]
    for r in range(rounds):
        torch.manual_seed(SEED + r)                      # each round samples different questions
        for i in range(0, len(prompts), batch_size):
            enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True).to("cuda")
            with torch.no_grad():
                out = llm.generate(**enc, max_new_tokens=60 * n, do_sample=True, temperature=0.8,
                                   top_p=0.9, pad_token_id=tok.pad_token_id)
            texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            for j, t in enumerate(texts):
                have = {q.lower() for q in results[i + j]}
                results[i + j] += [q for q in clean_lines(t, n) if q.lower() not in have]
            print(f"  round {r + 1}/{rounds}: {min(i + batch_size, len(prompts))}/{len(prompts)} passages")
    del llm; gc.collect(); torch.cuda.empty_cache()   # free the GPU for the embedding model
    return results

def glossary_pairs(glossary):
    """Each glossary term -> its definition, phrased a few ways (no language model needed)."""
    return [{"query": q, "chunk_id": f"glossary::{term}", "source": "glossary"}
            for term in glossary
            for q in (term, f"What does {term} mean?", f"define {term}", f"what is a {term}", f"{term} explained")]

# The saved library is tied to the documents and settings, so changing either writes a new one.
_fingerprint = hashlib.md5(json.dumps([GENERATOR_MODEL, [c["text"] for c in chunks]]).encode()).hexdigest()[:8]
PAIRS_CACHE = f"synthetic_library_{QUERIES_PER_CHUNK}x{QUESTION_ROUNDS}_{_fingerprint}.jsonl"

if os.path.exists(PAIRS_CACHE) and not REGENERATE:
    with open(PAIRS_CACHE) as f:
        generated = [json.loads(l) for l in f]
    print(f"Loaded the saved library: {len(generated)} questions from {PAIRS_CACHE}")
else:
    print(f"Writing questions for {len(chunks)} passages with {GENERATOR_MODEL} ...")
    qs = generate_queries([c["text"] for c in chunks])
    generated = [{"query": q, "chunk_id": c["id"], "source": "llm"} for c, ql in zip(chunks, qs) for q in ql]
    with open(PAIRS_CACHE, "w") as f:
        for row in generated: f.write(json.dumps(row) + "\n")
    print(f"Saved {len(generated)} questions to {PAIRS_CACHE}")

glossary_rows = glossary_pairs(glossary)
chunk_by_id = {c["id"]: c["text"] for c in chunks}
chunk_by_id.update({f"glossary::{t}": f"{t}: {d}" for t, d in glossary.items()})

per_passage = Counter(r["chunk_id"] for r in generated)
print(f"\n{len(generated)} written questions ({min(per_passage.values())}–{max(per_passage.values())} per passage) "
      f"+ {len(glossary_rows)} glossary pairs\n")
sample_rows = random.Random(SEED).sample(generated, min(10, len(generated)))
display(pd.DataFrame({"question": [r["query"] for r in sample_rows],
                      "answered by passage": [chunk_by_id[r["chunk_id"]][:90] + "…" for r in sample_rows]}))

## ▶ Step 3 · Load the base model, split the data and measure the starting point

**Split by passage, not by question.** Whole passages are held out, so test questions point at passages the model never trained on. At test time the model searches *all* passages, so it has to pick the right one out of everything.

- **Training set**: questions about the remaining passages, plus the glossary pairs. Each glossary term is also paired with the passage that uses it most, so it is learned in context, not just as a definition.
- **Validation set** (`VAL_FRACTION`): scored after every epoch in Step 4 to decide which epoch to keep.
- **Test set** (`TEST_FRACTION`): used only for the before and after scores, so choosing the epoch can't flatter the final result.

**Load the base model.** `BASE_MODEL` is loaded with a **LoRA** adapter attached: small extra matrices that are the only part trained. The adapter starts at zero, so the model still behaves exactly like the base model until training.

**Measure the starting point.** Every test question searches all passages; we record where the correct passage lands.

| Metric | Meaning |
|---|---|
| **Recall@k** | Share of questions whose correct passage is in the top *k* results. Recall@1 = right answer first. |
| **MRR@10** | Average of 1 ÷ rank of the correct passage (1st = 1.0, 2nd = 0.5, …; 0 if outside the top 10). |
| **NDCG@10** | Like MRR, with a gentler penalty for lower ranks. |

In [ ]:
# @title ▶ Step 3 · Load the base model, split the data and measure the starting point
# @markdown Share of passages held out for the final **test**, and for **validation** (choosing the best epoch)
TEST_FRACTION = 0.2  # @param {type:"slider", min:0.1, max:0.4, step:0.05}
VAL_FRACTION = 0.1  # @param {type:"slider", min:0.05, max:0.3, step:0.05}
# @markdown Longest passage the model reads (tokens), and the **LoRA rank** (adapter size)
MAX_SEQ_LEN = 384  # @param [256, 384, 512, 1024] {type:"raw"}
LORA_RANK = 32  # @param [8, 16, 32, 64] {type:"raw"}

if "generated" not in globals() or any(r["chunk_id"] not in chunk_by_id for r in generated):
    raise RuntimeError("The documents changed since Step 2 (or it hasn't run). Run Step 2 first.")

# ---- Split by passage -------------------------------------------------------------------
rng = random.Random(SEED)
chunk_ids = [c["id"] for c in chunks]
shuffled = rng.sample(chunk_ids, len(chunk_ids))
n_test = max(1, int(len(chunk_ids) * TEST_FRACTION))
n_val  = max(1, int(len(chunk_ids) * VAL_FRACTION))
test_ids, val_ids = set(shuffled[:n_test]), set(shuffled[n_test:n_test + n_val])
held_out = test_ids | val_ids

# Which document each passage belongs to; a glossary entry belongs to the document that uses the term most.
doc_of = {c["id"]: c["doc"] for c in chunks}
term_home = {t: home_chunk(t) for t in glossary}
doc_of.update({f"glossary::{t}": h["doc"] for t, h in term_home.items()})

# Glossary terms in context: the term also points at the passage that uses it (training passages only).
context_rows = [{"query": q, "chunk_id": h["id"], "source": "glossary-context"}
                for t, h in term_home.items() if h["id"] not in held_out
                for q in (t, f"What is {t}?")]

train_rows = [r for r in generated if r["chunk_id"] not in held_out] + glossary_rows + context_rows
val_rows   = [r for r in generated if r["chunk_id"] in val_ids]
test_rows  = [r for r in generated if r["chunk_id"] in test_ids]
corpus_ids   = chunk_ids                                   # search always runs over every passage
corpus_texts = [chunk_by_id[i] for i in corpus_ids]

# ---- Load the base model with a (still empty) LoRA adapter --------------------------------
def load_base_model():
    globals().pop("model", None); globals().pop("trainer", None)
    gc.collect(); torch.cuda.empty_cache()
    m = FastSentenceTransformer.from_pretrained(model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN,
                                                full_finetuning=False)
    return FastSentenceTransformer.get_peft_model(
        m, r=LORA_RANK, lora_alpha=LORA_RANK, lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=SEED, task_type="FEATURE_EXTRACTION")

model = load_base_model()
model_trained = False

# ---- Search and scoring helpers ------------------------------------------------------------
def embed(texts, is_query=False, batch_size=32):
    """🔌 The one function a RAG pipeline needs: text -> normalized vectors (queries get the instruction)."""
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float32):
        return model.encode(texts, prompt=QUERY_PROMPT if is_query else None, batch_size=batch_size,
                            convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)

def rank_metrics(ranked_ids, gold_ids, ks=(1, 3, 5, 10)):
    n = len(gold_ids)
    ranks = [r.index(g) + 1 if g in r else None for r, g in zip(ranked_ids, gold_ids)]
    m = {f"Recall@{k}": sum(1 for x in ranks if x and x <= k) / n for k in ks}
    m["MRR@10"]  = sum(1 / x for x in ranks if x and x <= 10) / n
    m["NDCG@10"] = sum(1 / math.log2(x + 1) for x in ranks if x and x <= 10) / n
    return m, ranks

def evaluate(rows, label=None):
    q = embed([r["query"] for r in rows], is_query=True)
    d = embed(corpus_texts)
    top = (q @ d.T).topk(min(10, len(corpus_ids)), dim=1).indices.cpu().tolist()
    metrics, ranks = rank_metrics([[corpus_ids[j] for j in row] for row in top], [r["chunk_id"] for r in rows])
    if label:
        print(f"{label:>14}: " + "  ".join(f"{k} {v:.3f}" for k, v in metrics.items()))
    return metrics, ranks

def top_hits(queries, k=3):
    scores = embed(queries, is_query=True) @ embed(corpus_texts).T
    top = scores.topk(min(k, len(corpus_ids)), dim=1)
    return [[(corpus_ids[i], round(s, 3)) for s, i in zip(vals, idxs)]
            for vals, idxs in zip(top.values.tolist(), top.indices.tolist())]

# ---- The starting point ---------------------------------------------------------------------
base_metrics, base_ranks = evaluate(test_rows)
if not EXAMPLE_QUERIES:
    EXAMPLE_QUERIES = [r["query"] for r in random.Random(SEED).sample(test_rows, min(4, len(test_rows)))]
examples_before = top_hits(EXAMPLE_QUERIES)

display(pd.DataFrame({"questions": [len(train_rows), len(val_rows), len(test_rows)],
                      "passages": [len(chunk_ids) - len(held_out), len(val_ids), len(test_ids)],
                      "used for": ["training (incl. glossary pairs)", "choosing the best epoch",
                                   "before / after scores"]},
                     index=["train", "validation", "test"]))
print(f"\nBase model {BASE_MODEL} on {len(test_rows)} held-out test questions:")
display(pd.DataFrame([base_metrics], index=["base model"]).round(3))

## ▶ Step 4 · Train

One press does everything from the untrained model to the final test score:

1. **Vocabulary snapshot.** A set of terms is embedded with the untrained model, so Step 5 can show how the model's view of your vocabulary changed. The terms are your glossary, `VOCAB_EXTRA`, and up to `N_AUTO_TERMS` phrases detected automatically: two-word phrases and capitalised names that recur in a few documents but aren't spread across all of them.
2. **Hard negatives.** For every training question, the wrong passage the base model ranks highest is attached as a deliberate trap. Two kinds of candidates are skipped, because they might actually be right: passages from **the same document** as the answer, and passages scoring almost as high as the answer.
3. **Fine-tuning.** `MultipleNegativesRankingLoss` pulls each question towards its passage and pushes it away from its hard negative **and** from every other passage in the same batch. Larger batches therefore mean more comparisons per step. Only the LoRA adapter is updated.
4. **Every epoch is scored** on the validation set and on the vocabulary terms. With `KEEP_BEST_EPOCH`, the best epoch is restored at the end, so a generous `EPOCHS` is safe.
5. **Final test.** The same held-out test questions from Step 3, now with the trained model.

**Reading the training curve.** Still rising at the last epoch → raise `EPOCHS`. Peaks early, then flat → more epochs won't help; more varied data will (Step 2). Jumpy → lower `LEARNING_RATE`.

You can change these settings and press ▶ again. The step starts from a fresh base model each time.

In [ ]:
# @title ▶ Step 4 · Train
# @markdown **Epochs** (upper limit: every epoch is scored and the best is kept)
EPOCHS = 10  # @param {type:"slider", min:1, max:30, step:1}
KEEP_BEST_EPOCH = True  # @param {type:"boolean"}
# @markdown **Learning rate** (1e-4 is typical for LoRA) and **batch size** (bigger = more comparisons per step)
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 32  # @param [16, 32, 64, 128] {type:"raw"}
# @markdown Phrases to detect automatically for the vocabulary map
N_AUTO_TERMS = 20  # @param {type:"slider", min:0, max:50, step:5}
MAX_TOPICS = 6                       # topics beyond this are shown as grey context on the map

import time
from datasets import Dataset
from transformers import TrainerCallback
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
try:
    from sentence_transformers import losses
    from sentence_transformers.training_args import BatchSamplers
except ImportError:                  # newer package layout
    from sentence_transformers.sentence_transformer import losses
    from sentence_transformers.base.sampler import BatchSamplers
import matplotlib.pyplot as plt

if "model" not in globals():
    raise RuntimeError("Run Step 3 first.")
if model_trained:                    # pressing ▶ again: start over from the base model
    print("Reloading the base model for a fresh run ...")
    model = load_base_model()
    model_trained = False

# ---- 1. Vocabulary snapshot (before training) ----------------------------------------------
def extract_terms(chunks, n_auto=N_AUTO_TERMS, exclude=()):
    """Two-word phrases and capitalised names that recur in a few documents but are not spread
    across all of them - a simple TF-IDF."""
    n_docs = len({c["doc"] for c in chunks})
    tf, df = Counter(), {}
    for c in chunks:
        w = _words(c["text"])
        grams = [f"{a} {b}" for a, b in zip(w, w[1:])
                 if a not in STOPWORDS and b not in STOPWORDS and len(a) > 2 and len(b) > 2]
        grams += [m for m in re.findall(r"(?<![.!?:]\s)(?<!^)\b([A-Z][a-z0-9-]{2,})\b", c["text"])
                  if m.lower() not in STOPWORDS]                 # capitalised mid-sentence
        tf.update(grams)
        for g in set(grams):
            df.setdefault(g, set()).add(c["doc"])
    excluded = [_key(e) for e in exclude]
    scored = []
    for g, n in tf.items():
        d, k = len(df[g]), _key(g)
        if n < 2 or d > max(3, 0.25 * n_docs) or any(k in e or e in k for e in excluded):
            continue
        scored.append((n * math.log(1 + n_docs / d) * (1.5 if " " in g else 1.0), g))
    picked = []
    for _, g in sorted(scored, reverse=True):
        if len(picked) >= n_auto:
            break
        if not any(_key(g) in _key(p) or _key(p) in _key(g) for p in picked):
            picked.append(g)
    return picked

vocab, seen = [], set()
for term, source in ([(t, "glossary") for t in glossary] + [(t, "extra") for t in VOCAB_EXTRA] +
                     [(t, "auto") for t in extract_terms(chunks, exclude=list(glossary) + VOCAB_EXTRA)]):
    if term.lower() not in seen:
        seen.add(term.lower())
        home = home_chunk(term)
        vocab.append({"term": term, "source": source, "home": home["id"], "home_doc": home["doc"],
                      "topic": DOC_GROUPS.get(home["doc"], home["doc"])})
top_topics = [t for t, _ in Counter(v["topic"] for v in vocab).most_common(MAX_TOPICS)]
for v in vocab:
    if v["topic"] not in top_topics:
        v["topic"] = "Other"
vocab_terms = [v["term"] for v in vocab]

def vocab_snapshot():
    return {"terms_doc":   embed(vocab_terms).float().cpu(),                 # word-to-word association
            "terms_query": embed(vocab_terms, is_query=True).float().cpu(),  # the term typed as a search
            "passages":    embed(corpus_texts).float().cpu()}

# Each term as a search: best passage of its home document vs best passage of any OTHER document.
corpus_docs = np.array([doc_of[i] for i in corpus_ids])
_same_doc = np.array([v["home_doc"] for v in vocab])[:, None] == corpus_docs[None, :]

def term_margins(terms_query, passages):
    Q = (terms_query @ passages.T).numpy()
    home  = np.where(_same_doc, Q, -np.inf).max(1)
    other = np.where(_same_doc, -np.inf, Q)
    return home - other.max(1), 1 + (other > home[:, None]).sum(1)   # margin, rank of the home document

vocab_before = vocab_snapshot()
margin_before, _ = term_margins(vocab_before["terms_query"], vocab_before["passages"])
print(f"Vocabulary snapshot: {len(vocab)} terms ({dict(Counter(v['source'] for v in vocab))})")

# ---- 2. Hard negatives -------------------------------------------------------------------------
def mine_hard_negatives(rows, margin=0.95):
    pool_ids = [i for i in chunk_by_id if i not in held_out]          # never use held-out passages
    pool_emb = embed([chunk_by_id[i] for i in pool_ids])
    sims = (embed([r["query"] for r in rows], is_query=True) @ pool_emb.T).cpu()
    index = {cid: i for i, cid in enumerate(pool_ids)}
    out = []
    for r, s in zip(rows, sims):
        pos, pos_doc = index[r["chunk_id"]], doc_of[r["chunk_id"]]
        ceiling = s[pos].item() * margin
        neg = next((j for j in s.argsort(descending=True).tolist()
                    if doc_of[pool_ids[j]] != pos_doc and s[j].item() < ceiling), None)
        if neg is None:
            neg = rng.choice([j for j in range(len(pool_ids)) if doc_of[pool_ids[j]] != pos_doc])
        out.append({"anchor": r["query"], "positive": chunk_by_id[r["chunk_id"]],
                    "negative": chunk_by_id[pool_ids[neg]]})
    return out

triplets = mine_hard_negatives(train_rows)
train_dataset = Dataset.from_list(triplets).shuffle(seed=SEED)
print(f"Hard negatives: {len(triplets)} (question, right passage, tempting wrong passage) triplets")

# ---- 3 + 4. Fine-tune, scoring every epoch -------------------------------------------------------
history, best = [], {"score": -float("inf")}

def lora_weights():
    return {n: p.detach().clone().cpu() for n, p in model.named_parameters() if p.requires_grad}

def track(epoch):
    metrics, _ = evaluate(val_rows, f"epoch {epoch} val")
    margins, _ = term_margins(embed(vocab_terms, is_query=True).float().cpu(), embed(corpus_texts).float().cpu())
    history.append({"epoch": epoch, **{f"val {k}": v for k, v in metrics.items()},
                    "vocab mean margin": margins.mean(), "vocab terms improved": (margins > margin_before).mean()})
    score = metrics["MRR@10"] + 1e-3 * margins.mean()        # validation first, vocabulary as tie-break
    if score > best["score"]:
        best.update(score=score, epoch=epoch, weights=lora_weights())

class EpochTracker(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        track(int(round(state.epoch)))
        model.train()

trainer = SentenceTransformerTrainer(
    model=model, train_dataset=train_dataset, loss=losses.MultipleNegativesRankingLoss(model),
    callbacks=[EpochTracker()],
    args=SentenceTransformerTrainingArguments(
        output_dir="checkpoints", num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE, warmup_ratio=0.1, lr_scheduler_type="cosine",
        fp16=not is_bf16_supported(), bf16=is_bf16_supported(),
        prompts={"anchor": QUERY_PROMPT},                     # the instruction goes on questions only
        batch_sampler=BatchSamplers.NO_DUPLICATES,            # never the same passage twice in a batch
        logging_steps=5, save_strategy="no", report_to="none", seed=SEED))

print(f"\nTraining for up to {EPOCHS} epochs ({len(triplets)} triplets, batch {BATCH_SIZE}) ...")
track(0)                                                      # the untrained model, for reference
model_trained = True
_t0 = time.time()
trainer.train()
train_minutes = (time.time() - _t0) / 60

last_epoch = history[-1]["epoch"]
if KEEP_BEST_EPOCH and best["epoch"] != last_epoch:
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in best["weights"]:
                p.copy_(best["weights"][n].to(p.device, p.dtype))
    kept_epoch = best["epoch"]
    print(f"\nKept epoch {kept_epoch} (best validation score); later epochs did not improve.")
else:
    kept_epoch = last_epoch
    print(f"\nKept the final epoch ({kept_epoch}).")
print(f"Trained in {train_minutes:.1f} min; peak GPU memory {torch.cuda.max_memory_reserved()/1024**3:.1f} GB")

# ---- Training curve ------------------------------------------------------------------------------
INK, INK_2, GRID, SURFACE, LINE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb", "#2a78d6"
curve = pd.DataFrame(history).set_index("epoch")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), facecolor=SURFACE)
for ax, col, title in [(axes[0], "val MRR@10", "Validation MRR@10 (higher is better)"),
                       (axes[1], "vocab mean margin", "Vocabulary: mean home-document margin")]:
    ax.set_facecolor(SURFACE)
    ax.plot(curve.index, curve[col], color=LINE, linewidth=2, marker="o", markersize=5,
            markeredgecolor=SURFACE, markeredgewidth=1.5)
    ax.axvline(kept_epoch, color=INK_2, linewidth=1, linestyle="--")
    ax.text(kept_epoch, 0.04, f" kept: epoch {kept_epoch}", transform=ax.get_xaxis_transform(),
            fontsize=8, color=INK_2, ha="left")
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.set_xlabel("epoch (0 = before training)", fontsize=9, color=INK_2)
    ax.set_xticks(curve.index)
    ax.tick_params(colors=INK_2, labelsize=8)
    ax.grid(axis="y", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_visible(False)
fig.tight_layout()
fig.savefig(f"{VIS_DIR}/training_curve.png", dpi=150, facecolor=SURFACE)
plt.show()

# ---- 5. Final test ---------------------------------------------------------------------------------
tuned_metrics, tuned_ranks = evaluate(test_rows)
results = pd.DataFrame([base_metrics, tuned_metrics], index=["base model", "fine-tuned"])
results.loc["change"] = results.loc["fine-tuned"] - results.loc["base model"]
print(f"Held-out test ({len(test_rows)} questions):")
display(results.round(3))

## ▶ Step 5 · Build the report

Collects everything into one document, `outputs/training_report.html`, shown below and downloaded together with the trained adapter:

- **Results at a glance**: the headline numbers before and after.
- **Did search improve?** Test scores, the questions that moved most, and example searches before vs after.
- **How training went**: the training curve and the epoch that was kept.
- **Did the vocabulary get organised?** Topic separation scores, a map of your terms per topic before vs after, each term's margin for finding its own document, an animated drift plot and nearest-word associations.
- **Why do this** and **what to do next**, with the caveats that apply to this run.

The vocabulary measures are computed in the full embedding space. The 2-D maps are projections: each stage is projected separately and then rotated to line up with the other, so treat distances on the maps as approximate.

In [ ]:
# @title ▶ Step 5 · Build the report
# @markdown How to flatten the vocabulary into 2-D: `tsne` shows neighbourhoods, `pca` keeps large-scale distances
PROJECTION = "tsne"  # @param ["tsne", "pca"]
# @markdown Save the trained LoRA adapter, and download everything as a zip
SAVE_ADAPTER = True  # @param {type:"boolean"}
DOWNLOAD_ZIP = True  # @param {type:"boolean"}
# @markdown Optional: also copy everything to this Google Drive folder (blank = don't)
COPY_TO_DRIVE = ""  # @param {type:"string"}

import base64, datetime, html as _html
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.linalg import orthogonal_procrustes

if not globals().get("model_trained"):
    raise RuntimeError("Run Step 4 first.")

# ---- Vocabulary: separation in the full embedding space ------------------------------------------
vocab_after = vocab_snapshot()
terms  = vocab_terms
topics = np.array([v["topic"] for v in vocab])

def separation(snap):
    X = snap["terms_doc"]
    S = (X @ X.T).numpy()
    keep = topics != "Other"
    pair = keep[:, None] & keep[None, :]
    same = (topics[:, None] == topics[None, :]) & ~np.eye(len(terms), dtype=bool) & pair
    diff = (topics[:, None] != topics[None, :]) & pair
    sil = (silhouette_score(X[keep].numpy(), topics[keep], metric="cosine")
           if len(set(topics[keep])) > 1 else float("nan"))
    margin, rank = term_margins(snap["terms_query"], snap["passages"])
    return {"S": S, "within": S[same].mean() if same.any() else float("nan"),
            "between": S[diff].mean() if diff.any() else float("nan"),
            "silhouette": sil, "margin": margin, "rank": rank}

sep_b, sep_a = separation(vocab_before), separation(vocab_after)
vocab_summary = pd.DataFrame([
    ("Topic silhouette (−1 to 1, higher = cleaner topics)", sep_b["silhouette"], sep_a["silhouette"]),
    ("Mean similarity within a topic",                       sep_b["within"], sep_a["within"]),
    ("Mean similarity between topics",                       sep_b["between"], sep_a["between"]),
    ("Separation gap (within − between)",                    sep_b["within"] - sep_b["between"],
                                                             sep_a["within"] - sep_a["between"]),
    ("Terms whose home document ranks #1",                   (sep_b["rank"] == 1).mean(), (sep_a["rank"] == 1).mean()),
    ("Mean margin (home document − best other document)",    sep_b["margin"].mean(), sep_a["margin"].mean()),
], columns=["measure", "before", "after"]).set_index("measure")
vocab_summary["change"] = vocab_summary["after"] - vocab_summary["before"]

per_term = pd.DataFrame({
    "term": terms, "topic": topics, "source": [v["source"] for v in vocab],
    "home document": [v["home_doc"] for v in vocab],
    "rank before": sep_b["rank"], "rank after": sep_a["rank"],
    "margin before": sep_b["margin"].round(3), "margin after": sep_a["margin"].round(3)})
per_term["margin change"] = (per_term["margin after"] - per_term["margin before"]).round(3)
per_term.to_csv(f"{OUT}/term_metrics.csv", index=False)

# ---- Charts --------------------------------------------------------------------------------------
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
HIGHLIGHT, CONTEXT = "#2a78d6", "#c9c8c2"

def project(X):
    X = X.numpy()
    if PROJECTION == "pca":
        P = PCA(n_components=2, random_state=SEED).fit_transform(X)
    else:
        P = TSNE(n_components=2, metric="cosine", init="pca", random_state=SEED,
                 perplexity=max(2, min(15, (len(X) - 1) // 3))).fit_transform(X)
    P = P - P.mean(0)
    return P / np.abs(P).max()

P_before = project(vocab_before["terms_doc"])
P_after  = project(vocab_after["terms_doc"])
R, _ = orthogonal_procrustes(P_after, P_before)     # rotate "after" to line up with "before"
P_after = P_after @ R
lim = np.abs(np.vstack([P_before, P_after])).max() * 1.25

def topic_within(S, idx):
    return S[np.ix_(idx, idx)][~np.eye(len(idx), dtype=bool)].mean() if len(idx) > 1 else float("nan")

fig, axes = plt.subplots(2, len(top_topics), figsize=(3.4 * len(top_topics), 7.4), squeeze=False, facecolor=SURFACE)
for r, (P, sep, label) in enumerate([(P_before, sep_b, "Before training"), (P_after, sep_a, "After training")]):
    for c, topic in enumerate(top_topics):
        ax = axes[r, c]
        ax.set_facecolor(SURFACE); ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_color(GRID)
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
        m = topics == topic
        ax.scatter(P[~m, 0], P[~m, 1], s=14, color=CONTEXT, linewidths=0)
        ax.scatter(P[m, 0], P[m, 1], s=44, color=HIGHLIGHT, edgecolors=SURFACE, linewidths=1.2, zorder=3)
        for (x, y), t in zip(P[m], np.array(terms)[m]):
            ax.annotate(t, (x, y), xytext=(4, 3), textcoords="offset points", fontsize=7.5, color=INK_2, zorder=4)
        ax.text(0.03, 0.03, f"within-topic similarity {topic_within(sep['S'], np.where(m)[0]):.2f}",
                transform=ax.transAxes, fontsize=8, color=INK_2)
        if r == 0:
            ax.set_title(topic, fontsize=11, color=INK, loc="left")
    axes[r, 0].set_ylabel(label, fontsize=11, color=INK)
fig.tight_layout()
fig.savefig(f"{VIS_DIR}/vocab_map_by_topic.png", dpi=150, facecolor=SURFACE)
plt.close(fig)

order = np.argsort(sep_a["margin"])
y = np.arange(len(terms))
fig, ax = plt.subplots(figsize=(8.5, 0.24 * len(terms) + 1.4), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.hlines(y, sep_b["margin"][order], sep_a["margin"][order], color=CONTEXT, linewidth=2, zorder=1)
ax.scatter(sep_b["margin"][order], y, s=36, color="#86b6ef", edgecolors=SURFACE, linewidths=1, zorder=2, label="Before training")
ax.scatter(sep_a["margin"][order], y, s=36, color="#1c5cab", edgecolors=SURFACE, linewidths=1, zorder=3, label="After training")
ax.axvline(0, color=INK_2, linewidth=1)
ax.set_yticks(y, np.array(terms)[order], fontsize=8, color=INK)
ax.tick_params(axis="x", colors=INK_2, labelsize=8); ax.tick_params(axis="y", length=0)
ax.grid(axis="x", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for s in ax.spines.values(): s.set_visible(False)
ax.set_xlabel("Similarity to the term's home document minus the best other document  (right of 0 = right document wins)",
              fontsize=9, color=INK_2)
ax.legend(loc="lower right", frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(f"{VIS_DIR}/term_margins.png", dpi=150, facecolor=SURFACE)
plt.close(fig)

points = [{"stage": stage, "term": v["term"], "topic": v["topic"], "x": P[i, 0], "y": P[i, 1],
           "home document rank": int(sep["rank"][i]),
           "change": "finds its document better" if per_term["margin change"][i] > 0 else "finds its document worse"}
          for stage, P, sep in [("before training", P_before, sep_b), ("after training", P_after, sep_a)]
          for i, v in enumerate(vocab)]
drift = px.scatter(pd.DataFrame(points), x="x", y="y", animation_frame="stage", animation_group="term",
                   color="change", text="term", hover_name="term",
                   hover_data={"topic": True, "home document rank": True, "x": False, "y": False,
                               "change": False, "stage": False},
                   color_discrete_map={"finds its document better": "#2a78d6", "finds its document worse": "#eb6834"},
                   range_x=[-lim, lim], range_y=[-lim, lim], height=700)
drift.update_traces(textposition="top center", textfont=dict(size=9, color=INK_2),
                    marker=dict(size=10, line=dict(width=1.5, color=SURFACE)))
drift.update_layout(plot_bgcolor=SURFACE, paper_bgcolor=SURFACE, font_color=INK, legend_title_text="",
                    margin=dict(l=10, r=10, t=30, b=10), legend=dict(orientation="h", y=1.04),
                    xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"))
drift.write_html(f"{VIS_DIR}/vocab_drift_interactive.html", include_plotlyjs="cdn")

def neighbours(S, i, k=3):
    return ", ".join([terms[j] for j in np.argsort(-S[i]) if j != i][:k])
by_change = per_term.sort_values("margin change")
pick = list(dict.fromkeys(list(by_change.index[::-1][:6]) + list(by_change.index[:3])))
associations = pd.DataFrame({"term": [terms[i] for i in pick],
                             "nearest terms before": [neighbours(sep_b["S"], i) for i in pick],
                             "nearest terms after": [neighbours(sep_a["S"], i) for i in pick],
                             "margin change": per_term["margin change"][pick].values})

# ---- Search examples and the questions that moved most -----------------------------------------
examples_after = top_hits(EXAMPLE_QUERIES)
def _rank_txt(r): return str(r) if r else ">10"
moves = pd.DataFrame({"question": [r["query"] for r in test_rows],
                      "answer passage": [r["chunk_id"] for r in test_rows],
                      "rank before": [_rank_txt(r) for r in base_ranks],
                      "rank after": [_rank_txt(r) for r in tuned_ranks],
                      "_delta": [(a or 11) - (b or 11) for a, b in zip(tuned_ranks, base_ranks)]})
improved = moves[moves["_delta"] < 0].sort_values("_delta").head(6).drop(columns="_delta")
worsened = moves[moves["_delta"] > 0].sort_values("_delta", ascending=False).head(4).drop(columns="_delta")

# ---- Save the adapter and the library ----------------------------------------------------------
ADAPTER_DIR = f"{OUT}/lora_adapter"
if SAVE_ADAPTER:
    model.save_pretrained(ADAPTER_DIR)
    model.tokenizer.save_pretrained(ADAPTER_DIR)
shutil.copy(PAIRS_CACHE, f"{OUT}/{os.path.basename(PAIRS_CACHE)}")

# ---- The report --------------------------------------------------------------------------------
esc = lambda s: _html.escape(str(s))
def img(path, alt):
    with open(path, "rb") as f:
        return f'<img src="data:image/png;base64,{base64.b64encode(f.read()).decode()}" alt="{esc(alt)}">'
def table(df, fmt="{:.3f}", index=True, header=True, signed_rows=(), signed_cols=()):
    df = df.copy()
    df.index.name = None
    for c in df.columns:
        if df[c].dtype.kind == "f":
            df[c] = [("{:+.3f}".format(v).replace("-", "−") if (r in signed_rows or c in signed_cols) else fmt.format(v))
                     for r, v in zip(df.index, df[c])]
    return df.to_html(index=index, header=header, escape=True, border=0, classes="t")
def pct(x): return f"{x * 100:.0f}%"
def arrow(d, eps): return "▲" if d > eps else ("▼" if d < -eps else "■")

b, a = base_metrics, tuned_metrics
d_r1, d_mrr = a["Recall@1"] - b["Recall@1"], a["MRR@10"] - b["MRR@10"]
gap_b, gap_a = vocab_summary.loc["Separation gap (within − between)", ["before", "after"]]
r1_b, r1_a = vocab_summary.loc["Terms whose home document ranks #1", ["before", "after"]]
n_terms_up = int((per_term["margin change"] > 0).sum())

if d_mrr > 0.02:
    verdict = (f"Search got better. On {len(test_rows)} questions about passages the model never trained on, "
               f"the right passage came first {pct(b['Recall@1'])} of the time before training and "
               f"{pct(a['Recall@1'])} after; MRR@10 went from {b['MRR@10']:.3f} to {a['MRR@10']:.3f}.")
elif d_mrr < -0.02:
    verdict = (f"Search got worse on the held-out test (MRR@10 {b['MRR@10']:.3f} → {a['MRR@10']:.3f}). "
               "This usually means the synthetic questions are too narrow or the model over-fitted them; "
               "see What next.")
else:
    verdict = (f"Search on the held-out test is about the same (MRR@10 {b['MRR@10']:.3f} → {a['MRR@10']:.3f}). "
               "The base model may already handle these documents well, or the test set is too small to show a change.")
def _moved(b_, a_, f):
    return f"stayed at {f(a_)}" if f(b_) == f(a_) else f"went from {f(b_)} to {f(a_)}"
vocab_text = (f"For the vocabulary, the separation gap between topics {_moved(gap_b, gap_a, lambda x: f'{x:.3f}')}, "
              f"the share of terms whose own document comes first {_moved(r1_b, r1_a, pct)}, and "
              f"{n_terms_up} of {len(per_term)} terms find their document more clearly than before.")
if kept_epoch == last_epoch and last_epoch == EPOCHS:
    curve_text = (f"The best validation score came at the last epoch ({kept_epoch} of {EPOCHS}), so the model "
                  "was still improving: try a higher EPOCHS.")
elif kept_epoch <= max(1, EPOCHS // 3):
    curve_text = (f"The validation score peaked early (epoch {kept_epoch} of {EPOCHS}) and later epochs did not "
                  "beat it, so more epochs won't help. More varied training data (more questions or rounds in "
                  "Step 2, more documents or glossary terms, a stronger question writer) is the next lever.")
else:
    curve_text = (f"The validation score peaked at epoch {kept_epoch} of {EPOCHS}, which was kept; "
                  "the run had enough epochs.")

kpis = [("Right passage ranked first", "held-out test, Recall@1", b["Recall@1"], a["Recall@1"], True),
        ("Mean reciprocal rank", "held-out test, MRR@10", b["MRR@10"], a["MRR@10"], False),
        ("Terms finding their own document", "vocabulary, ranked #1", r1_b, r1_a, True),
        ("Topic separation gap", "vocabulary, within − between", gap_b, gap_a, False)]
kpi_html = "".join(
    f'<div class="kpi"><div class="kpi-label">{esc(t)}</div><div class="kpi-sub">{esc(s)}</div>'
    f'<div class="kpi-val">{(pct(av) if isp else f"{av:.3f}")}</div>'
    f'<div class="kpi-delta">{arrow(av - bv, 0.005)} {"+" if av - bv >= 0 else "−"}'
    f'{(f"{abs(av - bv) * 100:.0f} pts" if isp else f"{abs(av - bv):.3f}")} '
    f'<span>from {(pct(bv) if isp else f"{bv:.3f}")}</span></div></div>'
    for t, s, bv, av, isp in kpis)

ex_rows = "".join(
    f"<tr><td>{esc(q)}</td><td>{esc(bh[0][0])} <span class='muted'>{bh[0][1]:.2f}</span></td>"
    f"<td>{esc(ah[0][0])} <span class='muted'>{ah[0][1]:.2f}</span></td></tr>"
    for q, bh, ah in zip(EXAMPLE_QUERIES, examples_before, examples_after))

settings = pd.DataFrame({"value": [
    BASE_MODEL, f"{len(docs)} documents → {len(chunks)} passages (≤ {CHUNK_MAX_WORDS} words)",
    f"{len(glossary)} terms", GENERATOR_MODEL,
    f"{len(generated)} questions ({QUERIES_PER_CHUNK} per passage × {QUESTION_ROUNDS} round(s))",
    f"{len(train_rows)} pairs → {len(triplets)} triplets with hard negatives",
    f"{len(val_rows)} validation / {len(test_rows)} test questions",
    f"LoRA rank {LORA_RANK}, lr {LEARNING_RATE:g}, batch {BATCH_SIZE}, up to {EPOCHS} epochs, kept epoch {kept_epoch}",
    f"{train_minutes:.1f} min on {GPU_NAME}"]},
    index=["Base model", "Documents", "Glossary", "Question writer", "Synthetic library",
           "Training data", "Held out", "Training", "Training time"])

DRIFT_SLOT = "<!--DRIFT-->"

REPORT_CSS = """
:root{--bg:#fcfcfb;--card:#ffffff;--ink:#0b0b0b;--ink2:#52514e;--muted:#7a7974;--line:#e4e3df;--accent:#2a78d6}
@media (prefers-color-scheme: dark){:root{--bg:#1a1a19;--card:#232322;--ink:#ffffff;--ink2:#c3c2b7;--muted:#9a998f;--line:#383835;--accent:#3987e5}}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.6 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
main{max-width:1040px;margin:0 auto;padding:40px 20px 80px}
h1{font-size:30px;line-height:1.2;margin:0 0 6px}
h2{font-size:21px;margin:48px 0 10px;padding-top:12px;border-top:1px solid var(--line)}
h3{font-size:16px;margin:24px 0 6px}
p,li{color:var(--ink2);max-width:76ch} .muted{color:var(--muted);font-size:13px}
.lede{font-size:17px;color:var(--ink2);max-width:72ch}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px;margin:18px 0}
.kpi{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:14px 16px}
.kpi-label{font-weight:600;font-size:14px}.kpi-sub{color:var(--muted);font-size:12px}
.kpi-val{font-size:28px;font-weight:650;margin-top:6px}.kpi-delta{font-size:13px;color:var(--ink2)}
.kpi-delta span{color:var(--muted)}
.flow{display:flex;flex-wrap:wrap;gap:8px;align-items:stretch;margin:16px 0}
.step{flex:1 1 150px;background:var(--card);border:1px solid var(--line);border-radius:10px;padding:10px 12px;font-size:13px;color:var(--ink2)}
.step b{display:block;color:var(--ink);font-size:14px;margin-bottom:2px}
table.t,table.x{border-collapse:collapse;width:100%;font-size:13px;margin:8px 0 4px;background:var(--card)}
table.t th,table.t td,table.x th,table.x td{border-bottom:1px solid var(--line);padding:6px 10px;text-align:left;vertical-align:top}
table.t th,table.x th{color:var(--ink2);font-weight:600}
.scroll{overflow-x:auto}
img{max-width:100%;height:auto;border:1px solid var(--line);border-radius:8px;background:#fcfcfb;margin:8px 0}
.callout{background:var(--card);border-left:3px solid var(--accent);padding:10px 14px;border-radius:4px;margin:14px 0}
"""

report_html = f"""<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1"><title>Embedding fine-tuning report</title>
<style>{REPORT_CSS}</style></head><body><main>
<h1>Embedding fine-tuning report</h1>
<p class="muted">{datetime.datetime.now():%Y-%m-%d %H:%M} · {esc(BASE_MODEL)} · {len(docs)} documents · kept epoch {kept_epoch}</p>
<p class="lede">{esc(verdict)}</p>

<h2>Results at a glance</h2>
<div class="kpis">{kpi_html}</div>
<p>{esc(vocab_text)}</p>

<h2>How this works: from pre-training to post-training</h2>
<p>An embedding model turns text into vectors so that texts with similar meaning sit close together; search returns the passages closest to the question. The base model learned "similar" from the public internet, so it has never seen this domain's terms. Fine-tuning shows it many examples of <i>this question is answered by this passage</i> from your own material, so its sense of similarity shifts towards what matters here.</p>
<div class="flow">
<div class="step"><b>1 · Data library</b>{len(docs)} documents split into {len(chunks)} passages, plus {len(glossary)} glossary terms.</div>
<div class="step"><b>2 · Synthetic library</b>A question writer produced {len(generated)} realistic questions; the glossary added {len(glossary_rows) + len(context_rows)} more pairs.</div>
<div class="step"><b>3 · Baseline</b>Passages were held out and the untrained model was scored on {len(test_rows)} questions about them.</div>
<div class="step"><b>4 · Training</b>A LoRA adapter learned from {len(triplets)} question / right passage / tempting wrong passage triplets; the best of {EPOCHS} epochs was kept.</div>
<div class="step"><b>5 · Re-measure</b>The same held-out questions and vocabulary were scored again.</div>
</div>
<div class="scroll">{table(settings, header=False)}</div>

<h2>Did search improve?</h2>
<p>Scores on held-out questions: the passages behind these questions were never used for training. Recall@k is the share of questions with the right passage in the top k; MRR@10 averages 1 ÷ rank.</p>
<div class="scroll">{table(results, signed_rows=("change",))}</div>
<h3>Example searches: top result before vs after</h3>
<div class="scroll"><table class="x"><tr><th>search</th><th>before training</th><th>after training</th></tr>{ex_rows}</table></div>
<h3>Questions that moved up most</h3>
<div class="scroll">{table(improved, index=False) if len(improved) else "<p class='muted'>None.</p>"}</div>
<h3>Questions that moved down</h3>
<div class="scroll">{table(worsened, index=False) if len(worsened) else "<p class='muted'>None — no held-out question got worse.</p>"}</div>

<h2>How training went</h2>
<p>{esc(curve_text)}</p>
{img(f"{VIS_DIR}/training_curve.png", "Training curve")}

<h2>Did the vocabulary get organised?</h2>
<p>{len(vocab)} terms were tracked: {len(glossary)} from the glossary, {sum(v['source'] == 'extra' for v in vocab)} added by hand and {sum(v['source'] == 'auto' for v in vocab)} detected in the documents. Each belongs to a topic and has a home document (the one that uses it most).</p>
<div class="scroll">{table(vocab_summary, signed_cols=("change",))}</div>
<p class="muted">A rising separation gap and silhouette mean a topic's terms pulled together <b>and</b> away from other topics. Within-topic similarity rising on its own isn't enough: if between-topic similarity rises as much, everything just got more alike.</p>
<h3>Map by topic</h3>
<p>One column per topic. Its terms are blue; every other term is grey for context. Before training, a topic's terms are usually scattered; after training they should sit closer together.</p>
{img(f"{VIS_DIR}/vocab_map_by_topic.png", "Vocabulary map by topic")}
<h3>Does each term find its own document?</h3>
<p>Each term typed as a search: how much more its home document scores than the best passage from any other document. Right of zero means the right document wins.</p>
{img(f"{VIS_DIR}/term_margins.png", "Term margins before and after")}
<h3>Vocabulary drift</h3>
<p>Press play to move every term from where the untrained model placed it to where the trained model does. Blue terms find their document better after training; orange ones worse.</p>
{DRIFT_SLOT}
<h3>Word associations</h3>
<p>The three nearest terms before and after, for the terms that changed most.</p>
<div class="scroll">{table(associations, index=False, signed_cols=("margin change",))}</div>

<h2>Why do this?</h2>
<ul>
<li><b>Search and RAG find the right passage for your terms.</b> A RAG assistant can only answer from what retrieval hands it. If the retriever misses the passage about a term, the assistant answers from nothing or from the wrong page.</li>
<li><b>Everyday wording maps to your jargon.</b> The synthetic library deliberately includes symptom-style questions that never name the term, which is how people actually search.</li>
<li><b>It's cheap.</b> LoRA trains about 1–2% of the weights; this run took {train_minutes:.1f} minutes on a {esc(GPU_NAME)}. The result is a normal embedding model you can serve anywhere (sentence-transformers, TEI, vLLM, a vector database).</li>
<li><b>It's measurable.</b> The held-out test and vocabulary measures show whether a change helped before it reaches users.</li>
</ul>

<h2>Caveats and what next</h2>
<ul>
<li><b>Small test set.</b> {len(test_rows)} test questions decide these scores; a few questions more or less can move them several points. A real evaluation needs a few hundred, ideally including real user questions.</li>
<li><b>Synthetic questions.</b> They were written by a {esc(GENERATOR_MODEL.split('/')[-1])} model, so they share its blind spots. A stronger writer, or a reviewed library, usually helps most.</li>
<li><b>Glossary terms were also trained on</b>, so part of the vocabulary improvement is expected. The held-out search scores are the test of generalisation.</li>
<li><b>Next steps:</b> add real questions to the test set; scale up the documents and question count; try a larger base model (Qwen3-Embedding-4B or 8B); re-embed your whole corpus with the new model before putting it into service.</li>
</ul>
<p class="muted">Files: training_report.html · charts/ · term_metrics.csv · lora_adapter/ · {esc(os.path.basename(PAIRS_CACHE))}</p>
</main></body></html>"""

# The saved report carries the plotting library inside it, so it also works offline;
# the copy shown in the notebook loads it from the web to keep the notebook small.
_drift = lambda js: drift.to_html(full_html=False, include_plotlyjs=js, config={"displayModeBar": False})
REPORT_PATH = f"{OUT}/training_report.html"
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(report_html.replace(DRIFT_SLOT, _drift(True)))

display(HTML(report_html.replace(DRIFT_SLOT, _drift("cdn"))))

if COPY_TO_DRIVE.strip():
    _mount_drive_if_needed(COPY_TO_DRIVE.strip())
    shutil.copytree(OUT, COPY_TO_DRIVE.strip(), dirs_exist_ok=True)
    print(f"Copied everything to {COPY_TO_DRIVE.strip()}")
if DOWNLOAD_ZIP:
    zip_path = shutil.make_archive("embedding_finetune_outputs", "zip", OUT)
    try:
        from google.colab import files
        files.download(zip_path)
    except ImportError:
        print(f"Saved {zip_path}")
print(f"Report saved to {REPORT_PATH}")

## Optional extras

These are not needed for the report. Use them to try searches, or to export the model for use elsewhere.

In [ ]:
# @title 🔎 Try a search with the trained model
QUERY = "robot stuck flashing amber and won't move"  # @param {type:"string"}
TOP_K = 3  # @param {type:"slider", min:1, max:10, step:1}
for cid, score in top_hits([QUERY], k=TOP_K)[0]:
    print(f"{score:.3f}  {cid}\n       {chunk_by_id[cid][:200]}\n")

In [ ]:
# @title 💾 Save the merged model (and optionally push it to the Hugging Face Hub)
# @markdown The merged model is a normal sentence-transformers model: load it anywhere without Unsloth.
SAVE_MERGED = True  # @param {type:"boolean"}
# @markdown Hub repo, e.g. `your-name/my-embedding-model` (blank = don't push; needs an `HF_TOKEN` secret)
HF_REPO = ""  # @param {type:"string"}
MERGED_DIR = f"{OUT}/merged_model"
if SAVE_MERGED:
    model.save_pretrained_merged(MERGED_DIR, tokenizer=model.tokenizer, save_method="merged_16bit")
    from sentence_transformers import SentenceTransformer
    try:
        st = SentenceTransformer(MERGED_DIR, device="cuda")
        q = st.encode([EXAMPLE_QUERIES[0]], prompt=QUERY_PROMPT, normalize_embeddings=True)
        d = st.encode(corpus_texts, normalize_embeddings=True)
        print(f"Saved to {MERGED_DIR}; reload check, top hit for '{EXAMPLE_QUERIES[0]}': "
              f"{corpus_ids[int((q @ d.T).argmax())]}")
    except Exception as e:
        print("Saved, but the reload check failed:", e)
if HF_REPO.strip():
    from google.colab import userdata
    model.push_to_hub_merged(HF_REPO.strip(), tokenizer=model.tokenizer, save_method="merged_16bit",
                             token=userdata.get("HF_TOKEN"))
    print(f"Pushed to https://huggingface.co/{HF_REPO.strip()}")

### Scaling up to Qwen3-Embedding-8B on bigger hardware

The same steps carry over. On, for example, 2 × 96 GB GPUs:

| Setting | Colab (T4) | 2 × 96 GB |
|---|---|---|
| `BASE_MODEL` | `unsloth/Qwen3-Embedding-0.6B` | `Qwen/Qwen3-Embedding-8B` |
| `GENERATOR_MODEL` | Qwen2.5-3B-Instruct | a much stronger model (local 30–70B, or a hosted API); question quality is the biggest lever |
| `QUERIES_PER_CHUNK` | 8 | 8–10, across thousands of passages |
| `MAX_SEQ_LEN` | 384 | 512–1024 |
| `BATCH_SIZE` | 32 | 128–512 per GPU; consider `CachedMultipleNegativesRankingLoss` for larger effective batches |
| `LORA_RANK` | 32 | 32–64 in bf16 (no 4-bit needed) |

LoRA on the 8B model fits on one 96 GB card. To use both cards with DDP, export the code of Steps 1–4 into `train.py` and run `accelerate launch --num_processes 2 train.py`, with `ddp_find_unused_parameters=False` in the training arguments and `gather_across_devices=True` on the loss so each GPU also uses the other GPU's batch as negatives.